# Индивидуальное задание для практики 7-8 (задание 3, вариант Б)

In [135]:
import sqlite3

In [136]:
db_name = "test.db"

In [137]:
conn = sqlite3.connect(db_name)

cursor = conn.cursor()

Создаем таблицы

In [138]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS employees (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        info BLOB DEFAULT NULL
    )
""")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS items (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        price REAL NOT NULL,
        employee_id INTEGER REFERENCES employees(id)
    )
""")
conn.commit()

Вставляем сотрудников

In [139]:
employees = []
employees.append(("Ivan", None))
employees.append(("Petr", "phone: 88888888888"))
for employee in employees:
    cursor.execute("INSERT INTO employees(name, info) VALUES(?, ?)", employee)

In [140]:
conn.commit()

In [141]:
cursor.execute("SELECT * FROM employees")
result = cursor.fetchall()
result

[(1, 'Ivan', None), (2, 'Petr', 'phone: 88888888888')]

Вставляем товары

In [142]:
items = []
items.append(("Phone", 105, 1))
items.append(("Car", 99999.435, None))
for item in items:
    cursor.execute("INSERT INTO items(name, price, employee_id) VALUES(?, ?, ?)", item)

In [143]:
conn.commit()

In [144]:
cursor.execute("SELECT * FROM items")
result = cursor.fetchall()
result

[(1, 'Phone', 105.0, 1), (2, 'Car', 99999.435, None)]

In [145]:
cursor.execute("SELECT * FROM items i JOIN employees e ON i.employee_id = e.id")
result = cursor.fetchall()
result

[(1, 'Phone', 105.0, 1, 1, 'Ivan', None)]

In [146]:
cursor.execute("DELETE FROM items WHERE name = 'Car'")
conn.commit()

In [147]:
cursor.execute("SELECT * FROM items")
result = cursor.fetchall()
result

[(1, 'Phone', 105.0, 1)]

Делаем GUI

In [148]:
from tkinter import Button, Entry, Label, StringVar, Tk, Toplevel, messagebox
from tkinter.ttk import Combobox, Treeview

CRUD функции

In [149]:
def refresh_employees():
    cursor.execute("SELECT name FROM employees ORDER BY name")
    names = [row[0] for row in cursor.fetchall()]
    combo["values"] = names
    if names and combo.get() not in names:
        combo.current(0)


def refresh_items():
    for row in tree.get_children():
        tree.delete(row)

    name = combo.get()
    if not name:
        return

    cursor.execute(
        """
        SELECT i.id, i.name, i.price
        FROM items i
        JOIN employees e ON i.employee_id = e.id
        WHERE e.name = ?
        ORDER BY i.name
    """,
        (name,),
    )

    for item_id, item_name, price in cursor.fetchall():
        tree.insert("", "end", iid=item_id, values=(item_name, f"{price:.2f}"))


def get_selected_employee_id():
    name = combo.get()
    if not name:
        return None
    cursor.execute("SELECT id FROM employees WHERE name = ?", (name,))
    row = cursor.fetchone()
    return row[0] if row else None


def show_items():
    refresh_items()


def open_item_dialog(item_id=None):
    employee_id = get_selected_employee_id()
    if not employee_id:
        messagebox.showwarning("Внимание", "Сначала выберите сотрудника")
        return

    dialog = Toplevel(window)
    dialog.title("Редактирование товара" if item_id else "Новый товар")
    dialog.geometry("350x200")
    dialog.transient(window)
    dialog.grab_set()  # модальное окно

    # Поля
    Label(dialog, text="Название:").grid(row=0, column=0, padx=10, pady=10, sticky="e")
    name_var = StringVar()
    Entry(dialog, textvariable=name_var, width=25).grid(
        row=0, column=1, padx=10, pady=10
    )

    Label(dialog, text="Цена:").grid(row=1, column=0, padx=10, pady=10, sticky="e")
    price_var = StringVar()
    Entry(dialog, textvariable=price_var, width=25).grid(
        row=1, column=1, padx=10, pady=10
    )

    if item_id:
        cursor.execute("SELECT name, price FROM items WHERE id = ?", (item_id,))
        row = cursor.fetchone()
        if row:
            name_var.set(row[0])
            price_var.set(str(row[1]))

    def save():
        name = name_var.get().strip()
        price_str = price_var.get().strip()

        # Валидация
        if not name:
            messagebox.showerror("Ошибка", "Название не может быть пустым")
            return
        try:
            price = float(price_str)
            if price < 0:
                raise ValueError
        except ValueError:
            messagebox.showerror("Ошибка", "Цена должна быть числом ≥ 0")
            return

        try:
            if item_id:
                # UPDATE
                cursor.execute(
                    "UPDATE items SET name = ?, price = ? WHERE id = ?",
                    (name, price, item_id),
                )
            else:
                # INSERT
                cursor.execute(
                    "INSERT INTO items (name, price, employee_id) VALUES (?, ?, ?)",
                    (name, price, employee_id),
                )
            conn.commit()
            refresh_items()
            dialog.destroy()
        except sqlite3.Error as e:
            messagebox.showerror("Ошибка БД", str(e))

    Button(dialog, text="Сохранить", command=save, width=15).grid(
        row=2, column=0, columnspan=2, pady=15
    )


def add_item():
    open_item_dialog(item_id=None)


def edit_item():
    selected = tree.selection()
    if not selected:
        messagebox.showwarning("Внимание", "Выберите товар для редактирования")
        return
    item_id = int(selected[0])
    open_item_dialog(item_id=item_id)


def delete_item():
    selected = tree.selection()
    if not selected:
        messagebox.showwarning("Внимание", "Выберите товар для удаления")
        return

    item_id = int(selected[0])

    # Получить имя для подтверждения
    cursor.execute("SELECT name FROM items WHERE id = ?", (item_id,))
    row = cursor.fetchone()
    if not row:
        return

    if messagebox.askyesno("Подтверждение", f"Удалить товар {row[0]}?"):
        cursor.execute("DELETE FROM items WHERE id = ?", (item_id,))
        conn.commit()
        refresh_items()

In [150]:
window = Tk()
window.title("Товары сотрудников")
window.geometry("600x500")

window.columnconfigure(0, weight=1)
window.columnconfigure(1, weight=2)
window.columnconfigure(2, weight=1)
window.rowconfigure(3, weight=1)

# Combobox сотрудников
Label(window, text="Сотрудник:").grid(row=0, column=1, sticky="w", pady=(10, 0))
combo = Combobox(window, width=40, state="readonly")
combo.grid(row=1, column=1, pady=5, sticky="ew")

# Таблица товаров
columns = ("name", "price")
tree = Treeview(window, columns=columns, show="headings", height=10)
tree.heading("name", text="Название")
tree.heading("price", text="Цена")
tree.column("name", width=300)
tree.column("price", width=100, anchor="e")
tree.grid(row=3, column=1, pady=10, sticky="nsew")

# Скроллбар
from tkinter import Scrollbar

scrollbar = Scrollbar(window, orient="vertical", command=tree.yview)
scrollbar.grid(row=3, column=2, sticky="ns", pady=10)
tree.configure(yscrollcommand=scrollbar.set)

# Кнопки
btn_frame = Label(window)
btn_frame.grid(row=4, column=1, pady=10)

Button(btn_frame, text="Показать", command=show_items, width=15).pack(
    side="left", padx=5
)
Button(btn_frame, text="Добавить", command=add_item, width=15).pack(side="left", padx=5)
Button(btn_frame, text="Редактировать", command=edit_item, width=15).pack(
    side="left", padx=5
)
Button(btn_frame, text="Удалить", command=delete_item, width=15).pack(
    side="left", padx=5
)

# Заполнить Combobox
refresh_employees()

combo.bind("<<ComboboxSelected>>", lambda e: refresh_items())

window.mainloop()
conn.close()

In [151]:
conn.close()  # Закрываем соединение

In [152]:
import os

if os.path.exists(db_name):
    os.remove(db_name)